In [1]:
import pandas as pd

path = r"C:\Users\Kunny\Downloads\fireprotdb_20251015-164116.csv"
df = pd.read_csv(path, low_memory=False)

In [2]:
core_cols = [
    # 식별
    # "EXPERIMENT_ID",
    # "SEQUENCE_ID",
    "PROTEIN",
    "ORGANISM",

    # 변이 정보
    "SUBSTITUTION",  
    "INSERTION",
    "DELETION",

    # DDG
    "DDG",

    # 실험 조건 (정제용)
    "PH",
    "EXP_TEMPERATURE",

    # 구조 / 서열 매핑
    "UNIPROTKB",
    "WWPDB",
    "MEGASCALE",

    # 출처
    "SOURCE_DATASET",
]

df = df[core_cols]

In [3]:
df["SOURCE_DATASET"].unique()

array(['ProTherm', 'Domainome FITNESS', 'Domainome DDG', 'MegaScale',
       'COZYME'], dtype=object)

In [4]:
df_missense = df[
    df["SUBSTITUTION"].notna() &
    df["INSERTION"].isna() &
    df["DELETION"].isna()
].copy()

df_missense = df_missense.drop(columns=["INSERTION", "DELETION"])

# 단일 missense만 (예: "A123V")
df_missense = df_missense[
    df_missense["SUBSTITUTION"].str.count(",") == 0
]

In [5]:
df_missense

,PROTEIN,ORGANISM,SUBSTITUTION,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,WWPDB,MEGASCALE,SOURCE_DATASET
108,Tryptophan synthase alpha chain,Escherichia coli (strain K12),S6P,NaN,7.2,NaN,P0A877,1WQ5,NaN,ProTherm
109,Tryptophan synthase alpha chain,Escherichia coli (strain K12),P21S,NaN,7.2,NaN,P0A877,1WQ5,NaN,ProTherm
110,Tryptophan synthase alpha chain,Escherichia coli (strain K12),F22S,NaN,7.2,NaN,P0A877,1WQ5,NaN,ProTherm
111,Tryptophan synthase alpha chain,Escherichia coli (strain K12),F22L,-2.2,7.8,25.0,P0A877,1WQ5,NaN,ProTherm
112,Tryptophan synthase alpha chain,Escherichia coli (strain K12),F22L,0.8,7.8,25.0,P0A877,1WQ5,NaN,ProTherm
...,...,...,...,...,...,...,...,...,...,...
5465655,Formate dehydrogenase,Candida boidinii,A141M,NaN,NaN,NaN,O13437,NaN,NaN,COZYME
5465656,Formate dehydrogenase,Candida boidinii,D149L,NaN,NaN,NaN,O13437,NaN,NaN,COZYME
5465657,Formate dehydrogenase,Candida boidinii,A264P,NaN,NaN,NaN,O13437,NaN,NaN,COZYME
5465658,Formate dehydrogenase,Candida boidinii,S333Y,NaN,NaN,NaN,O13437,NaN,NaN,COZYME


In [6]:
df_ddg = df_missense[df_missense["DDG"].notna()].copy()

In [7]:
df_ddg

,PROTEIN,ORGANISM,SUBSTITUTION,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,WWPDB,MEGASCALE,SOURCE_DATASET
111,Tryptophan synthase alpha chain,Escherichia coli (strain K12),F22L,-2.200000,7.8,25.0,P0A877,1WQ5,NaN,ProTherm
112,Tryptophan synthase alpha chain,Escherichia coli (strain K12),F22L,0.800000,7.8,25.0,P0A877,1WQ5,NaN,ProTherm
116,Tryptophan synthase alpha chain,Escherichia coli (strain K12),P28G,2.300000,7.0,25.0,P0A877,1WQ5,NaN,ProTherm
117,Tryptophan synthase alpha chain,Escherichia coli (strain K12),P28G,-0.700000,7.0,25.0,P0A877,1WQ5,NaN,ProTherm
118,Tryptophan synthase alpha chain,Escherichia coli (strain K12),P28G,1.660000,7.8,25.0,P0A877,1WQ5,NaN,ProTherm
...,...,...,...,...,...,...,...,...,...,...
834122,2M9E,NaN,S32Y,-0.241082,NaN,NaN,NaN,NaN,2M9E,MegaScale
834123,2M9E,NaN,S32F,-0.029135,NaN,NaN,NaN,NaN,2M9E,MegaScale
834124,2M9E,NaN,S32P,-0.154077,NaN,NaN,NaN,NaN,2M9E,MegaScale
5465546,FMN-dependent NADH:quinone oxidoreductase 1,Pseudomonas putida (strain ATCC 47054 / DSM 61...,Q192R,-1.700000,NaN,NaN,Q88IY3,NaN,NaN,COZYME


In [8]:
import re
import pandas as pd

def parse_sub(s):
    m = re.match(r"([A-Z])(\d+)([A-Z])", s)
    if m:
        return m.group(1), int(m.group(2)), m.group(3)
    return None, None, None

df_ddg[["WT", "POS", "MUT"]] = df_ddg["SUBSTITUTION"].apply(
    lambda x: pd.Series(parse_sub(x))
)

df_ddg = df_ddg.drop(columns=["SUBSTITUTION"])

df_ddg = df_ddg.dropna(subset=["WT", "POS", "MUT"])

In [9]:
df_ddg

,PROTEIN,ORGANISM,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,WWPDB,MEGASCALE,SOURCE_DATASET,WT,POS,MUT
111,Tryptophan synthase alpha chain,Escherichia coli (strain K12),-2.200000,7.8,25.0,P0A877,1WQ5,NaN,ProTherm,F,22,L
112,Tryptophan synthase alpha chain,Escherichia coli (strain K12),0.800000,7.8,25.0,P0A877,1WQ5,NaN,ProTherm,F,22,L
116,Tryptophan synthase alpha chain,Escherichia coli (strain K12),2.300000,7.0,25.0,P0A877,1WQ5,NaN,ProTherm,P,28,G
117,Tryptophan synthase alpha chain,Escherichia coli (strain K12),-0.700000,7.0,25.0,P0A877,1WQ5,NaN,ProTherm,P,28,G
118,Tryptophan synthase alpha chain,Escherichia coli (strain K12),1.660000,7.8,25.0,P0A877,1WQ5,NaN,ProTherm,P,28,G
...,...,...,...,...,...,...,...,...,...,...,...,...
834122,2M9E,NaN,-0.241082,NaN,NaN,NaN,NaN,2M9E,MegaScale,S,32,Y
834123,2M9E,NaN,-0.029135,NaN,NaN,NaN,NaN,2M9E,MegaScale,S,32,F
834124,2M9E,NaN,-0.154077,NaN,NaN,NaN,NaN,2M9E,MegaScale,S,32,P
5465546,FMN-dependent NADH:quinone oxidoreductase 1,Pseudomonas putida (strain ATCC 47054 / DSM 61...,-1.700000,NaN,NaN,Q88IY3,NaN,NaN,COZYME,Q,192,R


In [10]:
df_ddg["PDB_ID"] = df_ddg["WWPDB"].str.upper()
df_ddg = df_ddg.drop(columns=["WWPDB"])

In [11]:
import pandas as pd

# boolean masks
both_nan = df_ddg["UNIPROTKB"].isna() & df_ddg["PDB_ID"].isna()
only_uniprot = df_ddg["UNIPROTKB"].notna() & df_ddg["PDB_ID"].isna()
only_pdb = df_ddg["UNIPROTKB"].isna() & df_ddg["PDB_ID"].notna()
both_exist = df_ddg["UNIPROTKB"].notna() & df_ddg["PDB_ID"].notna()

# count summary
nan_summary = pd.DataFrame({
    "case": [
        "both_nan",
        "only_uniprot",
        "only_pdb",
        "both_exist"
    ],
    "count": [
        both_nan.sum(),
        only_uniprot.sum(),
        only_pdb.sum(),
        both_exist.sum()
    ]
})

nan_summary


,case,count
0,both_nan,402917
1,only_uniprot,2785
2,only_pdb,0
3,both_exist,6709


In [12]:
def source_stats(mask, name):
    return (
        df_ddg[mask]
        .groupby("SOURCE_DATASET")
        .size()
        .sort_values(ascending=False)
        .to_frame(name)
    )

stats_both_nan = source_stats(both_nan, "both_nan")
stats_only_uniprot = source_stats(only_uniprot, "only_uniprot")
stats_only_pdb = source_stats(only_pdb, "only_pdb")
stats_both_exist = source_stats(both_exist, "both_exist")

combined_stats = (
    pd.concat([
        stats_both_nan,
        stats_only_uniprot,
        stats_only_pdb,
        stats_both_exist
    ], axis=1)
    .fillna(0)
    .astype(int)
)

combined_stats

,both_nan,only_uniprot,only_pdb,both_exist
SOURCE_DATASET,,,,
MegaScale,402917,2520,0,0
ProTherm,0,263,0,6709
COZYME,0,2,0,0


In [13]:
#여기 수정 필요 Mega Scale보다 uniprotID를 먼저 잡아야 뒤에서 같은걸로 인식됨
#뒤에서도 평균 내기전에 mega scale보다 실제 데이터에 우선권을 주는것도 고려 필요(평균이 날수도?)

df_ddg["PROTEIN_ID"] = df_ddg["UNIPROTKB"]

mask_use_mega = (
    df_ddg["PROTEIN_ID"].isna() &
    (df_ddg["SOURCE_DATASET"] == "MegaScale") &
    df_ddg["MEGASCALE"].notna()
)

df_ddg.loc[mask_use_mega, "PROTEIN_ID"] = (
    df_ddg.loc[mask_use_mega, "MEGASCALE"].astype(str)
)

df_ddg = df_ddg.dropna(subset=["PROTEIN_ID"])

df_ddg["VARIANT_ID"] = (
    df_ddg["PROTEIN_ID"].astype(str) + "_" +
    df_ddg["WT"] + df_ddg["POS"].astype(str) + df_ddg["MUT"]
)

In [14]:
df_ddg

,PROTEIN,ORGANISM,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,MEGASCALE,SOURCE_DATASET,WT,POS,MUT,PDB_ID,PROTEIN_ID,VARIANT_ID
111,Tryptophan synthase alpha chain,Escherichia coli (strain K12),-2.200000,7.8,25.0,P0A877,NaN,ProTherm,F,22,L,1WQ5,P0A877,P0A877_F22L
112,Tryptophan synthase alpha chain,Escherichia coli (strain K12),0.800000,7.8,25.0,P0A877,NaN,ProTherm,F,22,L,1WQ5,P0A877,P0A877_F22L
116,Tryptophan synthase alpha chain,Escherichia coli (strain K12),2.300000,7.0,25.0,P0A877,NaN,ProTherm,P,28,G,1WQ5,P0A877,P0A877_P28G
117,Tryptophan synthase alpha chain,Escherichia coli (strain K12),-0.700000,7.0,25.0,P0A877,NaN,ProTherm,P,28,G,1WQ5,P0A877,P0A877_P28G
118,Tryptophan synthase alpha chain,Escherichia coli (strain K12),1.660000,7.8,25.0,P0A877,NaN,ProTherm,P,28,G,1WQ5,P0A877,P0A877_P28G
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
834122,2M9E,NaN,-0.241082,NaN,NaN,NaN,2M9E,MegaScale,S,32,Y,NaN,2M9E,2M9E_S32Y
834123,2M9E,NaN,-0.029135,NaN,NaN,NaN,2M9E,MegaScale,S,32,F,NaN,2M9E,2M9E_S32F
834124,2M9E,NaN,-0.154077,NaN,NaN,NaN,2M9E,MegaScale,S,32,P,NaN,2M9E,2M9E_S32P
5465546,FMN-dependent NADH:quinone oxidoreductase 1,Pseudomonas putida (strain ATCC 47054 / DSM 61...,-1.700000,NaN,NaN,Q88IY3,NaN,COZYME,Q,192,R,NaN,Q88IY3,Q88IY3_Q192R


In [15]:
df_ddg["HAS_PH"] = df_ddg["PH"].notna()
df_ddg["HAS_T"]  = df_ddg["EXP_TEMPERATURE"].notna()

group_I  = df_ddg[df_ddg["HAS_PH"] & df_ddg["HAS_T"]].copy()
group_II = df_ddg[~(df_ddg["HAS_PH"] & df_ddg["HAS_T"])].copy()

In [16]:
group_I["PH_DIST"] = (group_I["PH"] - 7.0).abs()
group_I["T_DIST"]  = (group_I["EXP_TEMPERATURE"] - 25).abs()

group_I["CORE_SCORE"] = group_I["PH_DIST"] + group_I["T_DIST"] / 20

In [17]:
core_samples = (
    group_I
    .sort_values("CORE_SCORE")
    .groupby("VARIANT_ID", as_index=False)
    .first()
)

In [18]:
core_samples

,VARIANT_ID,PROTEIN,ORGANISM,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,MEGASCALE,SOURCE_DATASET,WT,POS,MUT,PDB_ID,PROTEIN_ID,HAS_PH,HAS_T,PH_DIST,T_DIST,CORE_SCORE
0,O26594_V92L,None,Methanothermobacter thermautotrophicus (strain...,1.10,7.0,20.0,O26594,None,ProTherm,V,92,L,None,O26594,True,True,0.0,5.0,0.25
1,O49003_C450A,non-specific serine/threonine protein kinase,Avena sativa,-0.48,7.0,20.0,O49003,None,ProTherm,C,450,A,2V0U,O49003,True,True,0.0,5.0,0.25
2,O60885_A420D,Bromodomain-containing protein 4,Homo sapiens,-4.93,7.5,10.0,O60885,None,ProTherm,A,420,D,3MXF,O60885,True,True,0.5,15.0,1.25
3,O60885_A89V,Bromodomain-containing protein 4,Homo sapiens,-0.99,7.5,10.0,O60885,None,ProTherm,A,89,V,3MXF,O60885,True,True,0.5,15.0,1.25
4,O74035_D105A,Ribonuclease HII,Thermococcus kodakarensis (strain ATCC BAA-918...,-1.58,9.0,50.0,O74035,None,ProTherm,D,105,A,1IO2,O74035,True,True,2.0,25.0,3.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4431,R9S082_V115A,Myoglobin,Procyon lotor,1.40,9.6,82.2,R9S082,None,ProTherm,V,115,A,None,R9S082,True,True,2.6,57.2,5.46
4432,R9S082_V14A,Myoglobin,Procyon lotor,0.00,5.0,25.0,R9S082,None,ProTherm,V,14,A,None,R9S082,True,True,2.0,0.0,2.00
4433,R9S082_V69T,Myoglobin,Procyon lotor,0.20,7.8,4.0,R9S082,None,ProTherm,V,69,T,None,R9S082,True,True,0.8,21.0,1.85
4434,R9S082_W15F,Myoglobin,Procyon lotor,1.10,7.8,4.0,R9S082,None,ProTherm,W,15,F,None,R9S082,True,True,0.8,21.0,1.85


In [19]:
merged = []

for _, core in core_samples.iterrows():
    vid = core["VARIANT_ID"]
    ph0 = core["PH"]
    t0  = core["EXP_TEMPERATURE"]

    subset = group_I[
        (group_I["VARIANT_ID"] == vid) &
        (group_I["PH"].between(ph0 - 0.5, ph0 + 0.5)) &
        (group_I["EXP_TEMPERATURE"].between(t0 - 10, t0 + 10))
    ]
    merged.append(subset)

group_I_selected = pd.concat(merged).drop_duplicates()

In [20]:
group_I_selected

,PROTEIN,ORGANISM,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,MEGASCALE,SOURCE_DATASET,WT,POS,MUT,PDB_ID,PROTEIN_ID,VARIANT_ID,HAS_PH,HAS_T,PH_DIST,T_DIST,CORE_SCORE
94331,NaN,Methanothermobacter thermautotrophicus (strain...,1.10,7.0,20.0,O26594,NaN,ProTherm,V,92,L,NaN,O26594,O26594_V92L,True,True,0.0,5.0,0.25
84133,non-specific serine/threonine protein kinase,Avena sativa,-0.48,7.0,20.0,O49003,NaN,ProTherm,C,450,A,2V0U,O49003,O49003_C450A,True,True,0.0,5.0,0.25
77772,Bromodomain-containing protein 4,Homo sapiens,-4.93,7.5,10.0,O60885,NaN,ProTherm,A,420,D,3MXF,O60885,O60885_A420D,True,True,0.5,15.0,1.25
77773,Bromodomain-containing protein 4,Homo sapiens,-0.44,7.5,10.0,O60885,NaN,ProTherm,A,420,D,3MXF,O60885,O60885_A420D,True,True,0.5,15.0,1.25
77769,Bromodomain-containing protein 4,Homo sapiens,0.79,7.5,10.0,O60885,NaN,ProTherm,A,89,V,3MXF,O60885,O60885_A89V,True,True,0.5,15.0,1.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48699,Myoglobin,Procyon lotor,0.60,7.8,4.0,R9S082,NaN,ProTherm,W,15,F,NaN,R9S082,R9S082_W15F,True,True,0.8,21.0,1.85
48700,Myoglobin,Procyon lotor,0.50,7.8,4.0,R9S082,NaN,ProTherm,W,15,F,NaN,R9S082,R9S082_W15F,True,True,0.8,21.0,1.85
48677,Myoglobin,Procyon lotor,0.90,7.8,4.0,R9S082,NaN,ProTherm,W,8,F,NaN,R9S082,R9S082_W8F,True,True,0.8,21.0,1.85
48678,Myoglobin,Procyon lotor,-0.10,7.8,4.0,R9S082,NaN,ProTherm,W,8,F,NaN,R9S082,R9S082_W8F,True,True,0.8,21.0,1.85


In [21]:
selected_ids_I = set(group_I_selected["VARIANT_ID"].unique())

group_II_unique_only = group_II[
    ~group_II["VARIANT_ID"].isin(selected_ids_I)
].copy()

In [22]:
len(group_I_selected["VARIANT_ID"].unique()), len(group_II_unique_only["VARIANT_ID"].unique()) 

(4436, 347002)

In [23]:
df_step3 = pd.concat(
    [group_I_selected, group_II_unique_only],
    ignore_index=True
)

In [24]:
df_step3

,PROTEIN,ORGANISM,DDG,PH,EXP_TEMPERATURE,UNIPROTKB,MEGASCALE,SOURCE_DATASET,WT,POS,MUT,PDB_ID,PROTEIN_ID,VARIANT_ID,HAS_PH,HAS_T,PH_DIST,T_DIST,CORE_SCORE
0,NaN,Methanothermobacter thermautotrophicus (strain...,1.100000,7.0,20.0,O26594,NaN,ProTherm,V,92,L,NaN,O26594,O26594_V92L,True,True,0.0,5.0,0.25
1,non-specific serine/threonine protein kinase,Avena sativa,-0.480000,7.0,20.0,O49003,NaN,ProTherm,C,450,A,2V0U,O49003,O49003_C450A,True,True,0.0,5.0,0.25
2,Bromodomain-containing protein 4,Homo sapiens,-4.930000,7.5,10.0,O60885,NaN,ProTherm,A,420,D,3MXF,O60885,O60885_A420D,True,True,0.5,15.0,1.25
3,Bromodomain-containing protein 4,Homo sapiens,-0.440000,7.5,10.0,O60885,NaN,ProTherm,A,420,D,3MXF,O60885,O60885_A420D,True,True,0.5,15.0,1.25
4,Bromodomain-containing protein 4,Homo sapiens,0.790000,7.5,10.0,O60885,NaN,ProTherm,A,89,V,3MXF,O60885,O60885_A89V,True,True,0.5,15.0,1.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
410864,2M9E,NaN,-0.241082,NaN,NaN,NaN,2M9E,MegaScale,S,32,Y,NaN,2M9E,2M9E_S32Y,False,False,NaN,NaN,NaN
410865,2M9E,NaN,-0.029135,NaN,NaN,NaN,2M9E,MegaScale,S,32,F,NaN,2M9E,2M9E_S32F,False,False,NaN,NaN,NaN
410866,2M9E,NaN,-0.154077,NaN,NaN,NaN,2M9E,MegaScale,S,32,P,NaN,2M9E,2M9E_S32P,False,False,NaN,NaN,NaN
410867,FMN-dependent NADH:quinone oxidoreductase 1,Pseudomonas putida (strain ATCC 47054 / DSM 61...,-1.700000,NaN,NaN,Q88IY3,NaN,COZYME,Q,192,R,NaN,Q88IY3,Q88IY3_Q192R,False,False,NaN,NaN,NaN


In [25]:
df_step4 = (
    df_step3
    .groupby("VARIANT_ID", as_index=False)
    .agg(
        DDG_mean=("DDG", "mean"),
        DDG_std=("DDG", "std"),
        DDG_count=("DDG", "count"),
        SOURCE_DATASET=("SOURCE_DATASET", "first"),
        WT = ("WT", "first"),
        POS = ("POS", "first"),
        MUT = ("MUT", "first"),
        UNIPROTKB = ("UNIPROTKB", "first"),
        PDB_ID = ("PDB_ID", "first"),
        MEGASCALE = ("MEGASCALE", "first"),
        PROTEIN_ID = ("PROTEIN_ID", "first")
    )
)

In [26]:
df_step4

,VARIANT_ID,DDG_mean,DDG_std,DDG_count,SOURCE_DATASET,WT,POS,MUT,UNIPROTKB,PDB_ID,MEGASCALE,PROTEIN_ID
0,1A0N_A12C,-1.271679,0.125349,2,MegaScale,A,12,C,None,None,1A0N,1A0N
1,1A0N_A12D,-2.513681,0.083609,2,MegaScale,A,12,D,None,None,1A0N,1A0N
2,1A0N_A12E,-2.120722,0.229601,2,MegaScale,A,12,E,None,None,1A0N,1A0N
3,1A0N_A12F,-2.238222,0.068071,2,MegaScale,A,12,F,None,None,1A0N,1A0N
4,1A0N_A12G,-0.142810,0.021134,2,MegaScale,A,12,G,None,None,1A0N,1A0N
...,...,...,...,...,...,...,...,...,...,...,...,...
351433,v2_6IVS_Y7R,0.056541,NaN,1,MegaScale,Y,7,R,None,None,v2_6IVS,v2_6IVS
351434,v2_6IVS_Y7S,-0.514305,NaN,1,MegaScale,Y,7,S,None,None,v2_6IVS,v2_6IVS
351435,v2_6IVS_Y7T,-0.211927,NaN,1,MegaScale,Y,7,T,None,None,v2_6IVS,v2_6IVS
351436,v2_6IVS_Y7V,-0.062451,NaN,1,MegaScale,Y,7,V,None,None,v2_6IVS,v2_6IVS


In [27]:
sign_conflict_ids = (
    df_step3
    .groupby("VARIANT_ID")["DDG"]
    .apply(lambda x: (x > 0).any() and (x < 0).any())
)

bad_sign_ids = sign_conflict_ids[sign_conflict_ids].index

In [28]:
bad_std_ids = df_step4.loc[
    df_step4["DDG_std"] > 1.2, "VARIANT_ID"
]

In [29]:
len(bad_sign_ids), len(bad_std_ids)

(3810, 809)

In [30]:
bad_ids = set(bad_sign_ids) | set(bad_std_ids)

df_final = df_step4[
    ~df_step4["VARIANT_ID"].isin(bad_ids)
].copy()

In [31]:
len(df_step4), len(df_final), df_final["DDG_mean"].describe(), df_final["DDG_count"].describe()

(351438,
 347025,
 count    347025.000000
 mean         -0.720410
 std           1.038462
 min         -13.700000
 25%          -1.240083
 50%          -0.475106
 75%          -0.032859
 max          23.210000
 Name: DDG_mean, dtype: float64,
 count    347025.000000
 mean          1.152064
 std           0.477922
 min           1.000000
 25%           1.000000
 50%           1.000000
 75%           1.000000
 max          20.000000
 Name: DDG_count, dtype: float64)

In [32]:
df_final

,VARIANT_ID,DDG_mean,DDG_std,DDG_count,SOURCE_DATASET,WT,POS,MUT,UNIPROTKB,PDB_ID,MEGASCALE,PROTEIN_ID
0,1A0N_A12C,-1.271679,0.125349,2,MegaScale,A,12,C,None,None,1A0N,1A0N
1,1A0N_A12D,-2.513681,0.083609,2,MegaScale,A,12,D,None,None,1A0N,1A0N
2,1A0N_A12E,-2.120722,0.229601,2,MegaScale,A,12,E,None,None,1A0N,1A0N
3,1A0N_A12F,-2.238222,0.068071,2,MegaScale,A,12,F,None,None,1A0N,1A0N
4,1A0N_A12G,-0.142810,0.021134,2,MegaScale,A,12,G,None,None,1A0N,1A0N
...,...,...,...,...,...,...,...,...,...,...,...,...
351433,v2_6IVS_Y7R,0.056541,NaN,1,MegaScale,Y,7,R,None,None,v2_6IVS,v2_6IVS
351434,v2_6IVS_Y7S,-0.514305,NaN,1,MegaScale,Y,7,S,None,None,v2_6IVS,v2_6IVS
351435,v2_6IVS_Y7T,-0.211927,NaN,1,MegaScale,Y,7,T,None,None,v2_6IVS,v2_6IVS
351436,v2_6IVS_Y7V,-0.062451,NaN,1,MegaScale,Y,7,V,None,None,v2_6IVS,v2_6IVS


In [33]:
df_final = df_final[["PROTEIN_ID", "WT", "POS", "MUT", "DDG_mean", "SOURCE_DATASET", "UNIPROTKB", "PDB_ID", "MEGASCALE"]]

In [34]:
df_final

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE
0,1A0N,A,12,C,-1.271679,MegaScale,None,None,1A0N
1,1A0N,A,12,D,-2.513681,MegaScale,None,None,1A0N
2,1A0N,A,12,E,-2.120722,MegaScale,None,None,1A0N
3,1A0N,A,12,F,-2.238222,MegaScale,None,None,1A0N
4,1A0N,A,12,G,-0.142810,MegaScale,None,None,1A0N
...,...,...,...,...,...,...,...,...,...
351433,v2_6IVS,Y,7,R,0.056541,MegaScale,None,None,v2_6IVS
351434,v2_6IVS,Y,7,S,-0.514305,MegaScale,None,None,v2_6IVS
351435,v2_6IVS,Y,7,T,-0.211927,MegaScale,None,None,v2_6IVS
351436,v2_6IVS,Y,7,V,-0.062451,MegaScale,None,None,v2_6IVS


In [35]:
uniprot_nan_count = df_final["UNIPROTKB"].isna().sum()
uniprot_nan_ratio = df_final["UNIPROTKB"].isna().mean()

uniprot_nan_count, uniprot_nan_ratio

(341531, 0.9841682875873496)

In [36]:
megascale_nan_count = df_final["MEGASCALE"].isna().sum()
megascale_nan_ratio = df_final["MEGASCALE"].isna().mean()

megascale_nan_count, megascale_nan_ratio

(4308, 0.01241409120380376)

In [37]:
df_final.to_csv("fireprotdb_ddg_final_all_ver2.csv", index=False)

In [38]:
df_final

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE
0,1A0N,A,12,C,-1.271679,MegaScale,None,None,1A0N
1,1A0N,A,12,D,-2.513681,MegaScale,None,None,1A0N
2,1A0N,A,12,E,-2.120722,MegaScale,None,None,1A0N
3,1A0N,A,12,F,-2.238222,MegaScale,None,None,1A0N
4,1A0N,A,12,G,-0.142810,MegaScale,None,None,1A0N
...,...,...,...,...,...,...,...,...,...
351433,v2_6IVS,Y,7,R,0.056541,MegaScale,None,None,v2_6IVS
351434,v2_6IVS,Y,7,S,-0.514305,MegaScale,None,None,v2_6IVS
351435,v2_6IVS,Y,7,T,-0.211927,MegaScale,None,None,v2_6IVS
351436,v2_6IVS,Y,7,V,-0.062451,MegaScale,None,None,v2_6IVS


In [39]:
df_final[df_final["SOURCE_DATASET"]=="MegaScale"]

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE
0,1A0N,A,12,C,-1.271679,MegaScale,None,None,1A0N
1,1A0N,A,12,D,-2.513681,MegaScale,None,None,1A0N
2,1A0N,A,12,E,-2.120722,MegaScale,None,None,1A0N
3,1A0N,A,12,F,-2.238222,MegaScale,None,None,1A0N
4,1A0N,A,12,G,-0.142810,MegaScale,None,None,1A0N
...,...,...,...,...,...,...,...,...,...
351433,v2_6IVS,Y,7,R,0.056541,MegaScale,None,None,v2_6IVS
351434,v2_6IVS,Y,7,S,-0.514305,MegaScale,None,None,v2_6IVS
351435,v2_6IVS,Y,7,T,-0.211927,MegaScale,None,None,v2_6IVS
351436,v2_6IVS,Y,7,V,-0.062451,MegaScale,None,None,v2_6IVS


In [40]:
df_final[df_final["SOURCE_DATASET"]!="MegaScale"]

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE
304681,O26594,V,92,L,1.100000,ProTherm,O26594,None,None
304682,O49003,C,450,A,-0.480000,ProTherm,O49003,2V0U,None
304685,O61594,A,263,F,0.430000,ProTherm,O61594,5CG0,None
304686,O61594,D,260,A,0.670000,ProTherm,O61594,5CG0,None
304687,O61594,E,261,A,-0.530000,ProTherm,O61594,5CG0,None
...,...,...,...,...,...,...,...,...,...
310479,R9S082,T,68,A,0.300000,ProTherm,R9S082,None,None
310480,R9S082,V,115,A,1.400000,ProTherm,R9S082,None,None
310481,R9S082,V,14,A,0.000000,ProTherm,R9S082,None,None
310482,R9S082,V,69,T,0.400000,ProTherm,R9S082,None,None
